# 03 — Model Refinement: CV, Circuit Encoding, SC/VSC Recovery
`02_tire_degradation_model.ipynb` proved the leakage story and shipped a working
model. This notebook makes the model itself more rigorous, in three independent
steps, run in order because each one changes what "the model" means for the next:

1. **Rolling-origin CV hyperparameter search** — replace arbitrary `max_depth=12`
   with a tuned value, using a CV scheme that never violates the time-aware
   discipline from notebook 02.
2. **Circuit encoding** — replace the arbitrary alphabetical `circuit_code` integer
   with something that carries real information, computed without leaking target
   data across the split.
3. **SC/VSC recovery feature** — stop simply dropping safety-car-affected laps;
   give the model a signal for "how recently did track conditions change" instead.

Each section reports its own honest MAE against notebook 02's baseline, so you can
see which changes actually earned their complexity and which didn't.

In [ ]:
from pathlib import Path
import sys
import os

def find_backend_dir(start: Path) -> Path:
    """Walk upward from `start` until we find a directory named 'backend'
    that actually contains ml/, ingestion/, and .env.example — this makes
    the notebook runnable regardless of the cwd Jupyter was launched from
    (e.g. launched from repo root, from backend/, or from backend/notebooks/)."""
    candidates = [start] + list(start.parents)
    for p in candidates:
        maybe = p if p.name == "backend" else p / "backend"
        if (maybe / "ingestion").is_dir() and (maybe / ".env.example").exists():
            return maybe
    raise RuntimeError(
        f"Could not locate 'backend/' above {start}. "
        "Run this notebook from within the f1-strategy-lab repo."
    )

NOTEBOOK = Path.cwd()
BACKEND = find_backend_dir(NOTEBOOK)
PROJECT_ROOT = BACKEND.parent

# Allow imports like: from ml.tire_degradation.features import ...
sys.path.insert(0, str(BACKEND))

from dotenv import load_dotenv

env_path = BACKEND / ".env"
load_dotenv(env_path)

import pandas as pd
from sqlalchemy import create_engine, text
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

from ml.tire_degradation.features import build_training_frame

DATABASE_URL = os.getenv("DATABASE_URL")

if DATABASE_URL is None:
    raise RuntimeError(
        f"DATABASE_URL not found. Looked for it in {env_path}. "
        f"Does that file exist and does it define DATABASE_URL=...?"
    )

engine = create_engine(DATABASE_URL)
print(f"BACKEND resolved to: {BACKEND}")
print(f"Loaded .env from:    {env_path}  (exists: {env_path.exists()})")
print("Engine created OK.")

import matplotlib.pyplot as plt
import numpy as np
import itertools
plt.rcParams["figure.dpi"] = 110

from ml.tire_degradation.features import load_clean_laps, compute_baselines, build_circuit_map, KNOWN_COMPOUNDS


## Baseline recap
Load the same clean-laps frame and target as notebook 02, so every comparison below
is against a known reference point rather than a fresh, uncomparable number.

In [ ]:
raw = load_clean_laps(engine)
baselines = compute_baselines(raw)
data = raw.merge(baselines, on=["circuit", "compound"], how="inner")
data["lap_time_delta"] = data["lap_time_ms"] - data["baseline_ms"]

circuit_map_baseline = build_circuit_map(data)
data["circuit_code"] = data["circuit"].map(circuit_map_baseline)

compound_dummies = pd.get_dummies(data["compound"], prefix="compound")
for c in KNOWN_COMPOUNDS:
    col = f"compound_{c}"
    if col not in compound_dummies.columns:
        compound_dummies[col] = False

base_feature_cols = ["tire_age", "lap_number", "circuit_code", "ambient_temp_c", "track_temp_c"]
X_base = pd.concat([data[base_feature_cols], compound_dummies], axis=1).fillna(0)
y = data["lap_time_delta"]

train_mask = data["round"].isin(range(1, 16))
val_mask = data["round"].isin(range(16, 23))

baseline_model = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
baseline_model.fit(X_base[train_mask], y[train_mask])
baseline_mae = mean_absolute_error(y[val_mask], baseline_model.predict(X_base[val_mask]))
print(f"Notebook 02 baseline honest MAE (round 15/16 split, untuned params): {baseline_mae:.2f} ms")

---
## Part 1 — Rolling-origin CV hyperparameter search

**Why not plain `KFold`:** standard k-fold shuffles rows randomly across folds, which
re-introduces the exact same-race leakage that Step 4 of notebook 02 exposed — a fold
boundary that splits mid-race lets the model see 80% of a stint and predict the rest.

**What we use instead:** a rolling-origin scheme. Each fold trains on an *expanding*
window of early rounds and validates on the next couple of rounds — never on rounds
the model has already seen, and never shuffled. This is the same discipline as the
single 15/16 split in notebook 02, just repeated across several boundaries so a
hyperparameter choice isn't tuned to one arbitrary split.

In [ ]:
def rolling_splits(rounds_available, val_size=2, min_train_rounds=8):
    """Yield (train_rounds, val_rounds) tuples, expanding-window, no shuffling."""
    rounds_available = sorted(rounds_available)
    splits = []
    start = min_train_rounds
    while start + val_size <= len(rounds_available):
        train_rounds = rounds_available[:start]
        val_rounds = rounds_available[start:start + val_size]
        splits.append((train_rounds, val_rounds))
        start += val_size
    return splits

available_rounds = sorted(data["round"].unique())
splits = rolling_splits(available_rounds)
for tr, va in splits:
    print(f"train rounds {tr[0]}-{tr[-1]} ({len(tr)} rounds)  ->  val rounds {va}")

### Grid search
Small, deliberately — the point is to demonstrate the *correct* CV discipline, not to
exhaustively search a huge space. Extend the grid once this is running against real
data, if the winning region looks like it's at the edge of what's tested here.

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [8, 12, 16, None],
    "min_samples_leaf": [1, 3, 5],
}
grid = list(itertools.product(*param_grid.values()))
print(f"Grid size: {len(grid)} combinations x {len(splits)} folds = {len(grid)*len(splits)} fits")

results = []
for n_estimators, max_depth, min_samples_leaf in grid:
    fold_maes = []
    for train_rounds, val_rounds in splits:
        tr_mask = data["round"].isin(train_rounds)
        va_mask = data["round"].isin(val_rounds)
        m = RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            min_samples_leaf=min_samples_leaf, random_state=42, n_jobs=-1,
        )
        m.fit(X_base[tr_mask], y[tr_mask])
        fold_maes.append(mean_absolute_error(y[va_mask], m.predict(X_base[va_mask])))
    results.append({
        "n_estimators": n_estimators, "max_depth": max_depth, "min_samples_leaf": min_samples_leaf,
        "mean_mae": np.mean(fold_maes), "std_mae": np.std(fold_maes),
    })

cv_results = pd.DataFrame(results).sort_values("mean_mae")
cv_results.head(10)

**Stability matters as much as the mean.** A combination with the lowest mean MAE
but a high `std_mae` across folds is less trustworthy than a combination that's
slightly worse on average but consistent — it means the "best" number might just be
one lucky fold. Check both columns, not just the top row, before picking a winner.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(cv_results["mean_mae"], cv_results["std_mae"], alpha=0.5, s=25, color="#3b6fa0")
best = cv_results.iloc[0]
ax.scatter([best["mean_mae"]], [best["std_mae"]], color="#c0554a", s=80, zorder=5, label="lowest mean MAE")
ax.set_xlabel("mean MAE across folds (ms)")
ax.set_ylabel("std MAE across folds (ms)")
ax.set_title("Hyperparameter stability: mean vs. variance across rolling folds")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_params = cv_results.iloc[0][["n_estimators", "max_depth", "min_samples_leaf"]].to_dict()
best_params["n_estimators"] = int(best_params["n_estimators"])
best_params["min_samples_leaf"] = int(best_params["min_samples_leaf"])
if pd.notna(best_params["max_depth"]):
    best_params["max_depth"] = int(best_params["max_depth"])
else:
    best_params["max_depth"] = None
print("Selected hyperparameters:", best_params)

tuned_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
tuned_model.fit(X_base[train_mask], y[train_mask])
tuned_mae = mean_absolute_error(y[val_mask], tuned_model.predict(X_base[val_mask]))
print(f"Tuned model, same 15/16 split as baseline: {tuned_mae:.2f} ms  (baseline was {baseline_mae:.2f} ms)")

If `tuned_mae` isn't meaningfully better than `baseline_mae`, that's a legitimate,
reportable outcome — it means the untuned defaults were already close to as good as
this model family can do on this feature set, and effort is better spent on features
(Parts 2 and 3) than further hyperparameter search.

---
## Part 2 — Circuit encoding
The baseline `circuit_code` is alphabetical-order integers — the model has no way to
know Baku and Jeddah are both low-downforce street circuits while Barcelona is a
high-downforce permanent track. Two alternatives, compared honestly against each
other and against the baseline:

### 2a. Target encoding (out-of-fold)
Replace the circuit ID with **that circuit's own mean degradation rate**, computed
using *only training-round data* — never touching validation rounds — then applied to
validation. Circuits that appear only in validation (never seen in training) fall back
to the global training mean, since the model has no information about them otherwise.

This is a strictly train/val-safe version — computing the encoding from the full
dataset (train + val together) would leak target information across the split, which
is a real and easy-to-miss mistake with target encoding specifically.

In [ ]:
def circuit_target_encoding(train_df, all_circuits):
    circuit_means = train_df.groupby("circuit")["lap_time_delta"].mean()
    global_mean = train_df["lap_time_delta"].mean()
    return {c: circuit_means.get(c, global_mean) for c in all_circuits}

all_circuits = data["circuit"].unique()
te_map = circuit_target_encoding(data[train_mask], all_circuits)
data["circuit_target_enc"] = data["circuit"].map(te_map)

te_feature_cols = ["tire_age", "lap_number", "circuit_target_enc", "ambient_temp_c", "track_temp_c"]
X_te = pd.concat([data[te_feature_cols], compound_dummies], axis=1).fillna(0)

te_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
te_model.fit(X_te[train_mask], y[train_mask])
te_mae = mean_absolute_error(y[val_mask], te_model.predict(X_te[val_mask]))
print(f"Target-encoded circuit MAE: {te_mae:.2f} ms  (tuned baseline was {tuned_mae:.2f} ms)")

### 2b. Physical descriptors
A hand-coded downforce-level classification for the 2023 calendar, based on general,
well-known characteristics of each circuit (not from any single scraped or copyrighted
source). This is deliberately illustrative — it's a starting point, not something to
trust the way you'd trust real corner-speed telemetry. Flagged clearly as a limitation
below.

In [ ]:
# Illustrative classification only — replace with real circuit metadata
# (avg corner speed, downforce level, surface age) if this direction is pursued further.
DOWNFORCE_LEVEL = {
    "Sakhir": "medium", "Jeddah": "low", "Melbourne": "medium", "Baku": "low",
    "Miami": "medium", "Barcelona": "high", "Montréal": "medium", "Spielberg": "medium",
    "Silverstone": "high", "Budapest": "high", "Spa-Francorchamps": "low",
    "Monza": "low", "Marina Bay": "high", "Suzuka": "high", "Lusail": "high",
    "Austin": "medium", "Mexico City": "medium", "São Paulo": "medium",
    "Las Vegas": "low", "Yas Island": "medium", "Monaco": "high", "Zandvoort": "high",
}
downforce_rank = {"low": 0, "medium": 1, "high": 2}
data["downforce_rank"] = data["circuit"].map(DOWNFORCE_LEVEL).map(downforce_rank)
unmapped = data[data["downforce_rank"].isna()]["circuit"].unique()
if len(unmapped):
    print(f"WARNING — circuits missing from DOWNFORCE_LEVEL: {list(unmapped)}")

phys_feature_cols = ["tire_age", "lap_number", "downforce_rank", "ambient_temp_c", "track_temp_c"]
X_phys = pd.concat([data[phys_feature_cols], compound_dummies], axis=1).fillna(0)

phys_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
phys_model.fit(X_phys[train_mask], y[train_mask])
phys_mae = mean_absolute_error(y[val_mask], phys_model.predict(X_phys[val_mask]))
print(f"Physical-descriptor (downforce rank) MAE: {phys_mae:.2f} ms  (tuned baseline was {tuned_mae:.2f} ms)")

### Comparing all three circuit encodings side by side

In [ ]:
encoding_comparison = pd.DataFrame({
    "encoding": ["Arbitrary ID (baseline)", "Target encoding (OOF)", "Physical descriptor (downforce)"],
    "MAE": [tuned_mae, te_mae, phys_mae],
})

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(encoding_comparison["encoding"], encoding_comparison["MAE"], color=["#888888", "#3b6fa0", "#4a7a4a"])
ax.set_ylabel("MAE (ms)")
ax.set_title("Circuit encoding comparison (same tuned hyperparameters)")
plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
for i, v in enumerate(encoding_comparison["MAE"]):
    ax.text(i, v + 1, f"{v:.1f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

encoding_comparison

**How to read this honestly:** target encoding folds in real degradation-rate
information but does so using only 15 rounds' worth of per-circuit averages — with
that little data per circuit, it can be noisy in the other direction and risks
overfitting to whichever specific races were in the training window. The physical
descriptor is coarser (3 buckets vs. a continuous encoding) but doesn't depend on
having already seen degradation data for a circuit, which matters if this model ever
needs to make a first prediction for a circuit new to the calendar. Neither is a
strictly obvious win — pick based on which MAE actually came out lower **and** whether
generalizing to an unseen circuit matters for how this gets used.

---
## Part 3 — SC/VSC recovery feature
Notebook 02 filters to `track_status == '1'` and simply drops every non-green lap.
That throws away information: tires cool down significantly during a safety car
period, and the first few laps after a restart behave differently than steady-state
running. Instead of dropping this signal, derive it as a feature: **how many laps ago
was the most recent non-green track status**, computed per driver within a race, using
the *full* (unfiltered) lap history — then attach that feature to the clean laps used
for training.

In [ ]:
full_laps = pd.read_sql(text("""
    SELECT race_id, driver_id, lap_number, track_status
    FROM laps
    ORDER BY race_id, driver_id, lap_number
"""), engine)

def add_laps_since_sc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["race_id", "driver_id", "lap_number"]).copy()
    disrupted = df["track_status"] != "1"
    df["_disruption_lap"] = df["lap_number"].where(disrupted)
    df["_disruption_lap"] = df.groupby(["race_id", "driver_id"])["_disruption_lap"].ffill()
    # Laps before any disruption in the race get a sentinel (no SC/VSC yet, effectively "long ago")
    df["laps_since_sc"] = (df["lap_number"] - df["_disruption_lap"]).fillna(99)
    df["laps_since_sc"] = df["laps_since_sc"].clip(upper=20)  # cap the "long ago" tail
    return df.drop(columns="_disruption_lap")

full_laps = add_laps_since_sc(full_laps)
full_laps[["race_id", "driver_id", "lap_number", "track_status", "laps_since_sc"]].head(15)

In [ ]:
data_sc = data.merge(
    full_laps[["race_id", "driver_id", "lap_number", "laps_since_sc"]],
    on=["race_id", "driver_id", "lap_number"], how="left",
)
print(f"Rows with laps_since_sc attached: {data_sc['laps_since_sc'].notna().sum()} / {len(data_sc)}")

sc_feature_cols = ["tire_age", "lap_number", "circuit_code", "ambient_temp_c", "track_temp_c", "laps_since_sc"]
data_sc["circuit_code"] = data_sc["circuit"].map(circuit_map_baseline)
X_sc = pd.concat([data_sc[sc_feature_cols], compound_dummies.loc[data_sc.index]], axis=1).fillna({"laps_since_sc": 20})
X_sc = X_sc.fillna(X_sc.median(numeric_only=True))

sc_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
sc_model.fit(X_sc[train_mask], y[train_mask])
sc_mae = mean_absolute_error(y[val_mask], sc_model.predict(X_sc[val_mask]))
print(f"With laps_since_sc feature: {sc_mae:.2f} ms  (tuned baseline was {tuned_mae:.2f} ms)")

### Where does this feature actually help, if it does?
Averaging over the whole validation set can hide a real, localized effect. Check MAE
specifically on laps shortly after a restart (`laps_since_sc <= 3`) vs. steady-state
laps (`laps_since_sc >= 10`) — that's the more honest test of whether this feature is
doing anything, since it's designed to matter most exactly there.

In [ ]:
val_sc = data_sc[val_mask].copy()
val_sc["pred_with_sc"] = sc_model.predict(X_sc[val_mask])
val_sc["abs_error_with_sc"] = (val_sc["lap_time_delta"] - val_sc["pred_with_sc"]).abs()

val_sc["restart_bucket"] = pd.cut(
    val_sc["laps_since_sc"], bins=[-1, 3, 9, 20], labels=["just after SC (<=3)", "mid (4-9)", "steady-state (10+)"]
)
bucket_mae = val_sc.groupby("restart_bucket")["abs_error_with_sc"].agg(["mean", "count"]).rename(columns={"mean": "MAE"})

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(bucket_mae.index.astype(str), bucket_mae["MAE"], color="#3b6fa0")
ax.set_ylabel("MAE (ms)")
ax.set_title("MAE by proximity to last SC/VSC (with laps_since_sc feature)")
plt.setp(ax.get_xticklabels(), rotation=10, ha="right")
plt.tight_layout()
plt.show()

bucket_mae

If MAE is notably worse in the "just after SC" bucket even with this feature
included, that's not a failure of the feature — it's honest confirmation of the
documented limitation that safety-car-affected laps are inherently harder to predict,
and there just isn't much training data in that bucket to learn from (most laps are
steady-state by construction). Report the bucket breakdown either way.

---
## Part 4 — Combined stack: tuned hyperparameters + target-encoded circuit + laps_since_sc
Parts 1-3 each tested one change against the arbitrary-ID baseline in isolation.
Gains from separate feature changes do not reliably add up — this section actually
combines the winners and measures the real result, rather than assuming it.

In [ ]:
combined_feature_cols = ["tire_age", "lap_number", "circuit_target_enc", "ambient_temp_c", "track_temp_c", "laps_since_sc"]
X_combined = pd.concat([data_sc[combined_feature_cols], compound_dummies.loc[data_sc.index]], axis=1)
X_combined = X_combined.fillna(X_combined.median(numeric_only=True))

combined_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
combined_model.fit(X_combined[train_mask], y[train_mask])
combined_pred = combined_model.predict(X_combined[val_mask])
combined_mae = mean_absolute_error(y[val_mask], combined_pred)

print(f"Combined (tuned + target-encoded circuit + laps_since_sc): {combined_mae:.2f} ms")
print(f"Target encoding alone was:                                 {te_mae:.2f} ms")
print(f"laps_since_sc alone (on arbitrary ID) was:                  {sc_mae:.2f} ms")
print(f"Naive sum-of-improvements would predict:                   {tuned_mae - (tuned_mae - te_mae) - (tuned_mae - sc_mae):.2f} ms")

**How to read this:** if `combined_mae` is close to the "naive sum" line, the two
features are roughly independent and both earn their place. If it's notably worse
than that naive sum (regression toward `te_mae` alone), `laps_since_sc` is mostly
redundant with what circuit + tire_age already capture once circuit is a real
number instead of an arbitrary ID — worth reporting either way, since a small,
independently-real effect that disappears once combined is still a useful thing
to know before adding permanent feature-engineering complexity for it.

---
## Summary — what actually earned its complexity

In [ ]:
summary = pd.DataFrame({
    "change": [
        "Baseline (notebook 02, untuned params, arbitrary circuit ID)",
        "+ CV-tuned hyperparameters",
        "+ Target-encoded circuit (instead of arbitrary ID)",
        "+ Physical descriptor circuit (instead of arbitrary ID)",
        "+ laps_since_sc feature (on top of tuned + arbitrary ID)",
        "+ Combined: tuned + target-encoded circuit + laps_since_sc",
    ],
    "honest_MAE_ms": [baseline_mae, tuned_mae, te_mae, phys_mae, sc_mae, combined_mae],
})
summary["delta_vs_baseline"] = (summary["honest_MAE_ms"] - baseline_mae).round(2)
summary

Whatever this table actually shows once run against real data: report all five
rows in `ML_FINDINGS.md`, not just the winner. A change that made things worse is as
informative as one that helped — it tells the next person not to re-try it without a
different approach. Pick the combination with the best honest MAE **and** reasonable
stability (Part 1's std_mae logic applies here too, informally) to carry forward into
`features.py` / `train.py`.
